# Somo 02 - Kuchunguza Mfumo wa Microsoft Agent

**Microsoft Agent Framework (MAF)** ni mfumo uliounganishwa wa kujenga mawakala wa AI. Inatoa usanifu safi, unaoweza kuunganishwa na sehemu kuu nne za msingi:

- **Mteja** – huunganisha na kituo cha mfano wa AI na kushughulikia mawasiliano
- **Mwakala** – huvikwa mteja kwa maelekezo na ufafanuzi wa zana
- **Zana** – huongeza uwezo wa mwakala kwa kazi maalum ambazo mfano unaweza kuita
- **Kikao** – hudumisha historia ya mazungumzo kwa mwingiliano wa mizunguko mingi

Katika somo hili, tutajenga **mwakala wa uhifadhi wa usafiri** ambaye huangalia upatikanaji wa maeneo ya kwenda kwa kutumia dhana hizi.


## Usanidi


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Kuelewa Msingi wa Miundo ya Wakala

Mfumo wa Wakala wa Microsoft unafuata muundo wa tabaka:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Mteja** – `AzureAIProjectAgentProvider` huungana na usambazaji wa Azure OpenAI. Inashughulikia uthibitishaji, uundaji wa maombi, na uchambuzi wa majibu.
2. **Wakala** – Inaundwa kutoka kwa mteja kupitia `provider.create_agent()`, wakala huhusisha upatikanaji wa mfano pamoja na maelekezo (arifu ya mfumo) na zana.
3. **Zana** – Kazi za Python zilizo andikwa na `@tool` ambazo wakala anaweza kuzitumia kufanya vitendo au kupata data.
4. **Kikao** – Kitu cha `AgentSession` (kiliundwa kupitia `agent.create_session()`) kinachohifadhi historia ya mazungumzo, kuruhusu mazungumzo ya mizunguko mingi ambapo wakala anakumbuka muktadha wa awali.

Hebu tuunde kila tabaka hatua kwa hatua.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Kuongeza Zana kwa Mtafsiri wa @tool

Zana zinawawezesha mawakala kufanya vitendo zaidi ya kuandika maandishi. Mtafsiri wa `@tool` hubadilisha kazi ya kawaida ya Python kuwa kitu ambacho wakala anaweza kuitumia.

Vidokezo muhimu:
- Tumia `Annotated[type, "description"]` ili mfano ufahamu kila kipengele.
- Docstring hubadilika kuwa maelezo ya zana ambayo mfano unaiona.
- `approval_mode="never_require"` inamaanisha zana inafanya kazi moja kwa moja bila uthibitisho wa mtumiaji.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Kuunda Wakala na Zana

Sasa tunaunganisha mteja, maagizo, na zana kuwa wakala. `maagizo` hutumikia kama kichocheo cha mfumo — zinaelezea utu na tabia ya wakala.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Mazungumzo ya Mara Nyingi kwa Kikao

`AgentSession` (inayotengenezwa kupitia `agent.create_session()`) hufuatilia ujumbe wote katika mazungumzo. Kwa kupitisha kikao hicho hicho kwa kila simu ya `agent.run()`, wakala anaweza kupata historia kamili ya mazungumzo na kurejelea ujumbe wa awali.

Tunapita `tools=[check_destination_availability]` ili wakala aweze kutumia kaguzi wetu wa upatikanaji wakati wa kila zamu.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Muhtasari

Katika somo hili ulichunguza nguzo nne za Mfumo wa Wakala wa Microsoft:

| Dhana | Uliyojifunza |
|---------|------------------|
| **Mteja** | `AzureAIProjectAgentProvider` huunganishwa na Azure OpenAI kwa uthibitisho wa kipeperushi cha taarifa |
| **Wakala** | `provider.create_agent()` huunganisha muunganisho wa mfano na maagizo pamoja na jina |
| **Zana** | Mchapishaji wa `@tool` unaonyesha kazi za Python kwa wakala kuitumia |
| **Kikao** | `agent.create_session()` hudumisha historia ya mazungumzo kati ya mizunguko mingi |

Vipengele hivi vya msingi huunganishwa pamoja kuunda mawakala wanaoweza kuendesha mazungumzo asilia, kuita kazi za nje, na kudumisha muktadha — msingi wa mifumo ya wakala wa hali ya juu katika masomo yanayofuata.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Taarifa ya Kutotegemea**:
Hati hii imetafsiriwa kwa kutumia huduma ya tafsiri ya AI [Co-op Translator](https://github.com/Azure/co-op-translator). Ingawa tunajitahidi kwa usahihi, tafadhali fahamu kuwa tafsiri za moja kwa moja zinaweza kuwa na makosa au ukosefu wa usahihi. Hati ya asili katika lugha yake ya asili inapaswa kuchukuliwa kama chanzo halali. Kwa taarifa muhimu, tafsiri ya kitaalamu ya binadamu inapendekezwa. Hatuna dhamana kwa kutoelewana au tafsiri potofu zinazotokana na matumizi ya tafsiri hii.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
